In [2]:
import pandas as pd

BLACK_BELT_COUNTIES = [
    'Autauga', 'Barbour', 'Bullock', 'Butler', 'Choctaw', 'Clarke',
    'Crenshaw', 'Dallas', 'Greene', 'Hale', 'Lowndes', 'Macon',
    'Marengo', 'Monroe', 'Montgomery', 'Perry', 'Russell', 'Sumter', 'Wilcox'
]

def build_dataset(sos_filepath, cvap_filepath, l2_filepath, cvap_black, cvap_white, cvap_hisp, output_name):
    l2 = pd.read_csv(l2_filepath)
    l2['county'] = l2['countyname'].str.strip().str.title()
    l2['fips_code'] = l2['countyfips'].astype(str).str.zfill(5)
    fips_lookup = l2[l2['county'].isin(BLACK_BELT_COUNTIES)][['county', 'fips_code']]


    sos = pd.read_excel(sos_filepath, sheet_name='Table 1')
    sos.columns = sos.columns.str.replace('\n', ' ').str.strip()
    sos['county'] = sos['County'].astype(str).str.strip().str.title()
    sos = sos[sos['county'].isin(BLACK_BELT_COUNTIES)].copy()
    sos = sos.rename(columns={
        'Total Black (B)':    'voted_black',
        'Total White (W)':    'voted_white',
        'Total Hispanic (H)': 'voted_latinx',
    })[['county', 'voted_black', 'voted_white', 'voted_latinx']]

    cvap = pd.read_csv(cvap_filepath)
    cvap['fips_code'] = cvap['GEOID'].astype(str).str.zfill(5)
    fips_set = set(fips_lookup['fips_code'])
    cvap = cvap[cvap['fips_code'].isin(fips_set)].copy()
    cvap = cvap.rename(columns={
        cvap_black: 'eligible_black',
        cvap_white: 'eligible_white',
        cvap_hisp:  'eligible_latinx',
    })[['fips_code', 'eligible_black', 'eligible_white', 'eligible_latinx']]

    df = fips_lookup.merge(sos, on='county', how='inner')
    df = df.merge(cvap, on='fips_code', how='inner')

    for race in ['black', 'white', 'latinx']:
        df[f'missing_voters_{race}'] = (
            df[f'eligible_{race}'] - df[f'voted_{race}']
        ).apply(lambda x: max(0, x))

        df[f'missing_voters_rate_{race}'] = (
            df[f'missing_voters_{race}'] / df[f'eligible_{race}']
        ).round(4)

    df = df[[
        'county', 'fips_code',
        'eligible_black',   'voted_black',   'missing_voters_black',   'missing_voters_rate_black',
        'eligible_white',   'voted_white',   'missing_voters_white',   'missing_voters_rate_white',
        'eligible_latinx',  'voted_latinx',  'missing_voters_latinx',  'missing_voters_rate_latinx',
    ]]

    df = df.where(df.notna(), other='null')

    df.to_csv(output_name, index=False)
    print(f"Created: {output_name}")
    return df

df_2024 = build_dataset(
    sos_filepath  = '2024_race_gen_election.xlsx',
    cvap_filepath = 'al_cvap_2024_cnty.csv',
    l2_filepath   = 'AL_l2_2024_gen_stats_2020county.csv',
    cvap_black    = 'CVAP_BLA24',
    cvap_white    = 'CVAP_WHT24',
    cvap_hisp     = 'CVAP_HSP24',
    output_name   = 'voter_turnout_2024_general.csv'
)

df_2020 = build_dataset(
    sos_filepath  = '2020_race_gen_election.xlsx',
    cvap_filepath = 'al_cvap_2020_cnty.csv',
    l2_filepath   = 'AL_l2_2024_gen_stats_2020county.csv',
    cvap_black    = 'CVAP_BLK20',
    cvap_white    = 'CVAP_WHT20',
    cvap_hisp     = 'CVAP_HSP20',
    output_name   = 'voter_turnout_2020_general.csv'
)

df_2024

Created: voter_turnout_2024_general.csv
Created: voter_turnout_2020_general.csv


,county,fips_code,eligible_black,voted_black,missing_voters_black,missing_voters_rate_black,eligible_white,voted_white,missing_voters_white,missing_voters_rate_white,eligible_latinx,voted_latinx,missing_voters_latinx,missing_voters_rate_latinx
0,Autauga,01001,9023,5114,3909,0.4332,33452,22571,10881,0.3253,1199,225,974,0.8123
1,Barbour,01005,8929,3773,5156,0.5774,9301,5979,3322,0.3572,444,50,394,0.8874
2,Bullock,01011,5518,2966,2552,0.4625,1949,1099,850,0.4361,186,18,168,0.9032
3,Butler,01013,6207,2998,3209,0.5170,7580,5019,2561,0.3379,164,18,146,0.8902
4,Choctaw,01023,3722,2435,1287,0.3458,5637,4180,1457,0.2585,85,8,77,0.9059
5,Clarke,01025,7727,4749,2978,0.3854,9375,7118,2257,0.2407,127,20,107,0.8425
6,Crenshaw,01041,2425,1272,1153,0.4755,7218,5139,2079,0.2880,123,17,106,0.8618
7,Dallas,01047,19033,10418,8615,0.4526,8106,5191,2915,0.3596,165,13,152,0.9212
8,Greene,01063,4645,3175,1470,0.3165,1116,835,281,0.2518,4,12,0,0.0000
9,Hale,01065,6158,3803,2355,0.3824,4597,3465,1132,0.2462,105,17,88,0.8381
